# CNN-GNN Validation Experiment

- Keep the original test sites locked during experimentation.
- Use reproducible site-aware validation folds.
- Track the configuration and useful metrics in W&B.
- Save and upload only the best model.

## Setup

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import matplotlib.pyplot as plt
import pandas as pd
import torch

from abide_gnn import (
    BrainGraphClassifier,
    build_graphs,
    create_graph_loader,
    create_site_validation_folds,
    evaluate,
    fetch_cc200_atlas,
    initialize_wandb_run,
    load_abide_subjects,
    load_checkpoint,
    login_wandb,
    log_wandb_epoch,
    log_wandb_evaluation,
    predict,
    save_checkpoint,
    split_locked_test_sites,
    summarize_graphs,
    train_one_epoch,
)

### W&B Login

On the first online run, sign in when prompted. W&B stores the login outside this project and reuses it for later runs.

In [ ]:
wandb_authenticated = login_wandb()
print(f"W&B authenticated: {wandb_authenticated}")

## Configuration

W&B creates the run name automatically. `tags` contains the run tags, while `device` records which machine ran the experiment.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
ABIDE_DATA_DIR = DATA_DIR / "abide_pcp"
GRAPH_CACHE_DIR = DATA_DIR / "processed" / "graphs"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

config = {
    "data": {
        "n_subjects": None,
    },
    "graph": {
        "correlation_threshold": 0.5,
        "bold_normalization": "per_roi_zscore",
        "target_timepoints": 196,
        "negative_edge_policy": "signed",
    },
    "split": {
        "locked_test_sites": ["CALTECH", "CMU", "UM_2"],
        "validation_folds": 5,
        "validation_fold": 0,
        "seed": 42,
    },
    "loader": {
        "batch_size": 16,
    },
    "model": {
        "embedding_dim": 32,
        "hidden_dim": 64,
        "dropout": 0.3,
        "num_classes": 2,
    },
    "training": {
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "epochs": 50,
        "selection_metric": "roc_auc",
    },
    "wandb": {
        "project": "abide-gnn",
        "tags": ["fold-0"],
        "device": "Laptop",
    },
}

config

## Load Data and Build Graphs

In [ ]:
CC200_ATLAS_PATH = fetch_cc200_atlas(DATA_DIR)
subjects = load_abide_subjects(
    ABIDE_DATA_DIR,
    n_subjects=config["data"]["n_subjects"],
)
graphs = build_graphs(
    subjects,
    atlas_path=CC200_ATLAS_PATH,
    cache_dir=GRAPH_CACHE_DIR,
    **config["graph"],
)

pd.DataFrame(
    [
        {
            "Subject": graph.subject_id,
            "Site": graph.site_id,
            "Label": int(graph.y.item()),
            "BOLD": str(tuple(graph.bold.shape)),
            "Edges": graph.edge_index.shape[1],
            "Original time points": int(graph.original_timepoints.item()),
        }
        for graph in graphs
    ]
)

## Lock Test Sites and Select a Validation Fold

CALTECH, CMU, and UM_2 reproduce the original held-out test set. They are removed before the five development folds are created.

In [ ]:
development_graphs, locked_test_graphs = split_locked_test_sites(
    graphs,
    test_sites=config["split"]["locked_test_sites"],
)
validation_folds = create_site_validation_folds(
    development_graphs,
    n_splits=config["split"]["validation_folds"],
    seed=config["split"]["seed"],
)

fold_summary = pd.DataFrame(
    [
        {
            "Fold": fold_index,
            "Training sites": len({graph.site_id for graph in fold_train}),
            "Validation sites": len({graph.site_id for graph in fold_validation}),
            "Training participants": len(fold_train),
            "Validation participants": len(fold_validation),
            "Validation controls": sum(int(graph.y.item()) == 0 for graph in fold_validation),
            "Validation ASD": sum(int(graph.y.item()) == 1 for graph in fold_validation),
        }
        for fold_index, (fold_train, fold_validation) in enumerate(validation_folds)
    ]
).set_index("Fold")

fold_index = config["split"]["validation_fold"]
train_graphs, validation_graphs = validation_folds[fold_index]
train_loader = create_graph_loader(
    train_graphs,
    batch_size=config["loader"]["batch_size"],
    shuffle=True,
    seed=config["split"]["seed"],
)
validation_loader = create_graph_loader(
    validation_graphs,
    batch_size=config["loader"]["batch_size"],
)

active_split_summary = pd.DataFrame(
    {
        "Train": summarize_graphs(train_graphs),
        "Validation": summarize_graphs(validation_graphs),
        "Locked test": summarize_graphs(locked_test_graphs),
    }
).T

display(fold_summary, active_split_summary)

## Create the Baseline Model

W&B tracks the configuration, selected metrics, and system information. The run name is generated automatically.

In [ ]:
torch.manual_seed(config["split"]["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainGraphClassifier(**config["model"]).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)

wandb_run = initialize_wandb_run(config, ARTIFACTS_DIR)
RUN_DIR = ARTIFACTS_DIR / "experiments" / wandb_run.name
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / "best_model.pt"
wandb_run.config.update(
    {
        "train_sites": sorted({graph.site_id for graph in train_graphs}),
        "validation_sites": sorted({graph.site_id for graph in validation_graphs}),
    }
)

print(f"W&B run: {wandb_run.name}")
print(f"Run files: {RUN_DIR}")
model

## Train and Validate

W&B logs loss, accuracy, and ROC-AUC for training. Validation also includes balanced accuracy, ASD F1, sensitivity, and specificity.

In [ ]:
history = []
best_validation_roc_auc = float("-inf")

for epoch in range(1, config["training"]["epochs"] + 1):
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    validation_metrics = evaluate(
        model, validation_loader, criterion, device
    )

    epoch_record = {"epoch": epoch}
    epoch_record.update(
        {f"train_{name}": value for name, value in train_metrics.items()}
    )
    epoch_record.update(
        {
            f"validation_{name}": value
            for name, value in validation_metrics.items()
        }
    )
    history.append(epoch_record)

    is_best = validation_metrics["roc_auc"] > best_validation_roc_auc
    if is_best:
        best_validation_roc_auc = validation_metrics["roc_auc"]
        save_checkpoint(
            checkpoint_path=CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            config=config,
            validation_metrics=validation_metrics,
        )

    log_wandb_epoch(
        wandb_run,
        epoch,
        train_metrics,
        validation_metrics,
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train loss {train_metrics['loss']:.4f}, "
        f"accuracy {train_metrics['accuracy']:.3f}, "
        f"ROC-AUC {train_metrics['roc_auc']:.3f} | "
        f"validation loss {validation_metrics['loss']:.4f}, "
        f"accuracy {validation_metrics['accuracy']:.3f}, "
        f"balanced accuracy {validation_metrics['balanced_accuracy']:.3f}, "
        f"ROC-AUC {validation_metrics['roc_auc']:.3f}, "
        f"ASD F1 {validation_metrics['f1_asd']:.3f}, "
        f"sensitivity {validation_metrics['sensitivity_asd']:.3f}, "
        f"specificity {validation_metrics['specificity_control']:.3f}"
        f"{' | saved best' if is_best else ''}"
    )

## Training Curves

In [ ]:
history_frame = pd.DataFrame(history)
figure, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="Train")
axes[0].plot(
    history_frame["epoch"],
    history_frame["validation_loss"],
    label="Validation",
)
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[0].legend()

axes[1].plot(
    history_frame["epoch"],
    history_frame["train_roc_auc"],
    label="Train",
)
axes[1].plot(
    history_frame["epoch"],
    history_frame["validation_roc_auc"],
    label="Validation",
)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
axes[1].set(title="ROC-AUC", xlabel="Epoch", ylabel="ROC-AUC", ylim=(0, 1))
axes[1].legend()

plt.tight_layout()
plt.show()

## Finish the Validation Run

W&B receives the best validation metrics, a confusion matrix, an ROC curve, and the best model.

In [ ]:
checkpoint = load_checkpoint(CHECKPOINT_PATH, model, device=device)
best_validation_metrics = evaluate(
    model,
    validation_loader,
    criterion,
    device,
)
validation_predictions = predict(model, validation_loader, device)

log_wandb_evaluation(
    wandb_run,
    predictions=validation_predictions,
    best_epoch=checkpoint["epoch"],
    validation_metrics=best_validation_metrics,
    checkpoint_path=CHECKPOINT_PATH,
)

wandb_run_name = wandb_run.name
wandb_run_url = wandb_run.url
wandb_run.finish()

pd.Series(
    {
        "Run": wandb_run_name,
        "W&B run": wandb_run_url or "disabled/offline",
        "Best epoch": checkpoint["epoch"],
        "Validation accuracy": best_validation_metrics["accuracy"],
        "Validation ROC-AUC": best_validation_metrics["roc_auc"],
        "Validation balanced accuracy": best_validation_metrics["balanced_accuracy"],
        "Validation ASD sensitivity": best_validation_metrics["sensitivity_asd"],
        "Validation control specificity": best_validation_metrics["specificity_control"],
        "Validation ASD F1": best_validation_metrics["f1_asd"],
        "Confusion matrix [[TN, FP], [FN, TP]]": best_validation_metrics["confusion_matrix"],
        "Best model": str(CHECKPOINT_PATH),
    }
)

## Locked Test Set

The 60 participants from CALTECH, CMU, and UM_2 are listed in the split summary but are not loaded into a test data loader and are not evaluated here. For each candidate configuration, run validation folds 0 through 4 and compare the mean and spread of the fold-level metrics. Evaluate the locked test participants once only after choosing the final configuration.